### **1. Problem Setup & Data Split**

Se predice 'quality' (variable objetivo, escala 3-8) del dataset *Red Wine Quality* (1599 muestras, 11 features fisicoquímicos). Se usa **un único split 80/20** con 'random_state' **fijo** para que todos los modelos se evalúen sobre exactamente el mismo test set. Esto garantiza una comparación justa.



In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

RANDOM_STATE = 42

df = pd.read_csv("winequality-red.csv")
X = df.drop(columns="quality")
y = df["quality"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

### **2. Evaluation Helper**

In [ ]:
def evaluate(model, X_te, y_te, name):
    """Compute regression metrics on the test set.

    Args:
        model: Fitted estimator exposing a ``predict`` method.
        X_te: Test features.
        y_te: Test target.
        name: Label used in the comparison table.

    Returns:
        dict: Model name, RMSE, MAE and R2.
    """
    y_pred = model.predict(X_te)
    return {
        "Model": name,
        "RMSE": np.sqrt(mean_squared_error(y_te, y_pred)),
        "MAE": mean_absolute_error(y_te, y_pred),
        "R2": r2_score(y_te, y_pred),
    }

### **3. Model Training: Baselines vs. Ensemble**

El SVR se envuelve en un 'Pipeline' con 'StandardScaler' para evitar **data leakage**: el scaler se ajusta solo con 'X_train'. El árbol y el Random Forest no requieren feature scaling, ya que sus splits son invariantes a transformaciones monótonas.

In [ ]:
tree = DecisionTreeRegressor(max_depth=5, random_state=RANDOM_STATE)
svr = make_pipeline(StandardScaler(), SVR(kernel="rbf", C=1.0))
rf = RandomForestRegressor(
    n_estimators=300,
    max_features="sqrt",  # decorrelates trees (random feature subsets per split)
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

for model in (tree, svr, rf):
    model.fit(X_train, y_train)

results = pd.DataFrame([
    evaluate(tree, X_test, y_test, "Decision Tree"),
    evaluate(svr, X_test, y_test, "SVR (RBF)"),
    evaluate(rf, X_test, y_test, "Random Forest"),
]).round(4)
print(results)

           Model    RMSE     MAE      R2
0  Decision Tree  0.6584  0.4964  0.3366
1      SVR (RBF)  0.5928  0.4535  0.4623
2  Random Forest  0.5419  0.4150  0.5506


### **4. Key Insights**

* **Performance ranking**: Random Forest > SVR > Decision Tree en las tres métricas. El RMSE de 0.54 indica que, en promedio, la predicción se desvía ~0.5 puntos de una escala de calidad de 3 a 8.
* **Variance reduction**: el Random Forest promedia árboles entrenados sobre muestras bootstrap y subconjuntos aleatorios de features. Al decorrelacionar los árboles, el promedio reduce la varianza del árbol único sin aumentar mucho el bias.
* **R² interpretation**: el Random Forest explica ~55% de la varianza de quality. Hay margen de mejora, y el siguiente paso natural es el hyperparameter tuning (RandomizedSearchCV sobre max_features, min_samples_leaf, n_estimators).
* **Performance vs. interpretability trade-off**: el Random Forest es el mejor modelo en métricas, pero el menos interpretable. Un árbol de profundidad 5 es visualizable y explicable a negocio. El bosque requiere herramientas post-hoc como feature_importances_.
* **Caveat**: como el target es ordinal y discreto, una variante de clasificación u ordinal regression podría ser una alternativa válida.

### **Exercise 2: XGBoost vs. Classical Classifiers on UCI Credit Card Default**
## **1. Problem Setup & Data Split**
El objetivo es predecir default.'payment.next.month' (clasificación binaria). La clase positiva representa ~22% de las observaciones, por lo que usamos un **stratified split** para preservar la proporción en train y test. Todos los modelos se evalúan sobre el **mismo split** para garantizar una comparación justa.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score)

RANDOM_STATE = 42
TARGET = "default.payment.next.month"

df = pd.read_csv("UCI_Credit_Card.csv")
X = df.drop(columns=["ID", TARGET])
y = df[TARGET]

# Stratify to keep the ~22% positive-class ratio in both splits
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

                 Model  Accuracy  Precision  Recall      F1  ROC_AUC
0  Logistic Regression    0.8077     0.6868  0.2396  0.3553   0.7076
1        Random Forest    0.8135     0.6368  0.3647  0.4638   0.7548
2              XGBoost    0.8177     0.6572  0.3670  0.4710   0.7750


### **2. Evaluation Helper**


In [ ]:
def evaluate(model, X_te, y_te, name):
    """Compute classification metrics on the test set.

    Args:
        model: Fitted classifier exposing predict and predict_proba.
        X_te: Test features.
        y_te: Test target.
        name: Label for the model.

    Returns:
        dict with Model, Accuracy, Precision, Recall, F1 and ROC_AUC.
    """
    y_pred = model.predict(X_te)
    y_proba = model.predict_proba(X_te)[:, 1]  # ROC-AUC needs probabilities, not labels
    return {
        "Model": name,
        "Accuracy": accuracy_score(y_te, y_pred),
        "Precision": precision_score(y_te, y_pred),
        "Recall": recall_score(y_te, y_pred),
        "F1": f1_score(y_te, y_pred),
        "ROC_AUC": roc_auc_score(y_te, y_proba),
    }

In [ ]:
# Scaling only for Logistic Regression; tree ensembles are scale-invariant.
# The pipeline fits the scaler on X_train only, which avoids data leakage.
logreg = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
)
rf = RandomForestClassifier(
    n_estimators=300, max_features="sqrt",
    random_state=RANDOM_STATE, n_jobs=-1,
)
xgb = XGBClassifier(
    n_estimators=300, learning_rate=0.06, max_depth=5,
    subsample=0.8, colsample_bytree=0.8,
    eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=-1,
)

for model in (logreg, rf, xgb):
    model.fit(X_train, y_train)

results = pd.DataFrame([
    evaluate(logreg, X_test, y_test, "Logistic Regression"),
    evaluate(rf, X_test, y_test, "Random Forest"),
    evaluate(xgb, X_test, y_test, "XGBoost"),
]).round(4)
print(results)

### **3. Model Training & Comparison**


### **4. Key Insights**

Ranking global: XGBoost obtiene el mejor Accuracy, Recall, F1 y ROC-AUC. La logística solo lidera en Precision (0.6868), pero a costa de un Recall muy bajo (0.2396): es conservadora y marca pocos casos como default.
Magnitud de la mejora: El salto grande ocurre entre el modelo lineal y los ensambles (Recall 0.24 → 0.37). Entre XGBoost y Random Forest la diferencia es modesta (ROC-AUC +0.02, F1 +0.007).
Bagging vs. Boosting: Random Forest promedia árboles independientes y reduce principalmente varianza. XGBoost entrena árboles de forma secuencial, donde cada uno corrige los errores del anterior, y reduce principalmente bias. Esto explica su ventaja en ROC-AUC.
Accuracy es engañoso: Con ~78% de clase mayoritaria, un clasificador trivial ya alcanzaría ~0.78 de Accuracy. Las métricas relevantes para el negocio son Recall, F1 y ROC-AUC.
Interpretability trade-off: La logística es la más explicable (coeficientes) pero la menos performante. XGBoost es una black box; su interpretabilidad se recupera parcialmente con feature_importances_ o SHAP.